# IF Task-Vector Grad-Norm Top-k Sparse Updates (1%, 10%, 50%, 100%)

This notebook creates four checkpoints by filtering IF task-vector coordinates
with global magnitude thresholds and applying sparse updates to the base model:

- `theta_sparse = theta_base + mask * (theta_if - theta_base)`
- `mask = 1[score >= threshold]`
- `score = |theta_if - theta_base|` (coordinate-wise IF task-vector magnitude)

Saved variants use keep ratios: **top 1%**, **top 10%**, **top 50%**, **top 100%**.

In [ ]:
from __future__ import annotations

import gc
import json
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Mapping, Tuple

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from transformers import AutoModelForCausalLM, AutoTokenizer


@dataclass(frozen=True)
class RuntimeConfig:
    """Runtime configuration for IF sparse task-vector updates.

    Args:
        base_model_id: Base model id/path used as the merge anchor.
        if_model_path: IF-tuned model checkpoint path that defines the IF task vector.
        output_root: Root output directory for sparse checkpoints and metadata.
        model_dtype_name: Loading dtype alias (`bf16`, `fp16`, or `fp32`).
        seed: RNG seed used for threshold sampling reproducibility.
    """

    base_model_id: str
    if_model_path: Path
    output_root: Path
    model_dtype_name: str
    seed: int


RUNTIME = RuntimeConfig(
    base_model_id='Qwen/Qwen3-1.7B',
    if_model_path=Path('/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-ifrl_ifeval/global_step_50/actor/huggingface'),
    output_root=Path('/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-fisher-merge'),
    model_dtype_name='bf16',
    seed=42,
)

# Requested keep ratios: top 1%, top 10%, top 50%, top 100%.
SPARSE_KEEP_RATIOS: Tuple[float, ...] = (0.01, 0.10, 0.50, 1.00)

# We estimate global thresholds from a bounded sample for memory efficiency.
THRESHOLD_SAMPLE_SIZE = 2_000_000

# Dedicated output namespace for this notebook's artifacts.
SPARSE_OUTPUT_ROOT = RUNTIME.output_root / 'if_sparse_topk_task_vector_gradnorm'
SUMMARY_PATH = RUNTIME.output_root / 'metadata' / 'if_sparse_topk_task_vector_gradnorm_summary.json'

if not RUNTIME.if_model_path.exists():
    raise FileNotFoundError(f'IF checkpoint path does not exist: {RUNTIME.if_model_path}')

SPARSE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)

print(f'Base model: {RUNTIME.base_model_id}')
print(f'IF model path: {RUNTIME.if_model_path}')
print(f'Output root: {SPARSE_OUTPUT_ROOT}')
print(f'Summary path: {SUMMARY_PATH}')

In [ ]:
def now_iso() -> str:
    """Return a UTC ISO-8601 timestamp string.

    Returns:
        Timestamp string in `YYYY-MM-DDTHH:MM:SSZ` format.
    """

    return datetime.utcnow().isoformat(timespec='seconds') + 'Z'


def to_json_compatible(obj: Any) -> Any:
    """Recursively convert runtime objects into JSON-serializable values.

    Args:
        obj: Arbitrary Python object possibly containing `Path` or `torch.dtype`.

    Returns:
        JSON-safe nested structure.
    """

    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, torch.dtype):
        return str(obj)
    if isinstance(obj, dict):
        return {str(key): to_json_compatible(value) for key, value in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [to_json_compatible(value) for value in obj]
    return obj


def save_json(payload: Mapping[str, Any], output_path: Path) -> None:
    """Save JSON payload with conversion for runtime-only object types.

    Args:
        payload: Mapping payload to serialize.
        output_path: Destination JSON path.

    Returns:
        None. File is written to disk.
    """

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open('w', encoding='utf-8') as file:
        json.dump(to_json_compatible(dict(payload)), file, indent=2, ensure_ascii=False)


def resolve_torch_dtype(dtype_name: str) -> torch.dtype:
    """Resolve a dtype alias into an actual torch dtype.

    Args:
        dtype_name: One of `bf16`, `fp16`, or `fp32`.

    Returns:
        Torch dtype object.
    """

    lookup = {
        'bf16': torch.bfloat16,
        'fp16': torch.float16,
        'fp32': torch.float32,
    }
    if dtype_name not in lookup:
        raise ValueError(f'Unsupported dtype: {dtype_name}')
    return lookup[dtype_name]


def load_tokenizer_with_mistral_regex_fix(model_name_or_path: str) -> AutoTokenizer:
    """Load tokenizer with optional compatibility argument.

    Args:
        model_name_or_path: HF id or local checkpoint directory.

    Returns:
        Loaded tokenizer instance.
    """

    try:
        return AutoTokenizer.from_pretrained(
            model_name_or_path,
            trust_remote_code=True,
            fix_mistral_regex=True,
        )
    except TypeError:
        return AutoTokenizer.from_pretrained(
            model_name_or_path,
            trust_remote_code=True,
        )


def load_causal_lm(
    model_name_or_path: str | Path,
    torch_dtype: torch.dtype,
    device: str,
) -> tuple[AutoModelForCausalLM, AutoTokenizer]:
    """Load causal LM and tokenizer on the requested device.

    Args:
        model_name_or_path: HF id or local checkpoint path.
        torch_dtype: Loading dtype.
        device: Device string (typically `cpu` for this notebook).

    Returns:
        Tuple of `(model, tokenizer)`.
    """

    resolved = str(model_name_or_path)
    model = AutoModelForCausalLM.from_pretrained(
        resolved,
        torch_dtype=torch_dtype,
        device_map=None,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    )
    model.to(device)
    model.eval()

    tokenizer = load_tokenizer_with_mistral_regex_fix(resolved)
    if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
        tokenizer.pad_token = tokenizer.eos_token

    return model, tokenizer


def validate_parameter_compatibility(
    base_model: AutoModelForCausalLM,
    if_model: AutoModelForCausalLM,
) -> None:
    """Validate that base and IF checkpoints expose identical float-parameter layouts.

    Args:
        base_model: Base checkpoint model.
        if_model: IF checkpoint model.

    Returns:
        None. Raises `ValueError` on incompatible parameter names/shapes.
    """

    base_named = dict(base_model.named_parameters())
    if_named = dict(if_model.named_parameters())

    if set(base_named.keys()) != set(if_named.keys()):
        missing_in_if = sorted(set(base_named.keys()) - set(if_named.keys()))
        missing_in_base = sorted(set(if_named.keys()) - set(base_named.keys()))
        raise ValueError(
            'Parameter key mismatch between base and IF models. ' 
            f'missing_in_if={missing_in_if[:5]}, missing_in_base={missing_in_base[:5]}'
        )

    for param_name, base_param in base_named.items():
        if if_named[param_name].shape != base_param.shape:
            raise ValueError(
                f"Shape mismatch at '{param_name}': "
                f"base={tuple(base_param.shape)}, if={tuple(if_named[param_name].shape)}"
            )


def build_if_task_vector_gradnorm_scores(
    base_model: AutoModelForCausalLM,
    if_model: AutoModelForCausalLM,
) -> Dict[str, torch.Tensor]:
    """Build coordinate-wise IF task-vector grad-norm proxy scores.

    Score definition:
        score_j = |Delta_if_j|
        Delta_if_j = theta_if_j - theta_base_j

    Rationale:
        For scalar coordinates, L2 norm magnitude and absolute value are identical.
        Using `|Delta_if|` gives a direct coordinate-level update strength signal for
        top-k filtering before sparse reconstruction tests.

    Args:
        base_model: Base model that defines `theta_base`.
        if_model: IF model that defines `theta_if`.

    Returns:
        Mapping `parameter_name -> score_tensor` on CPU float32.
    """

    validate_parameter_compatibility(base_model=base_model, if_model=if_model)

    base_named = dict(base_model.named_parameters())
    if_named = dict(if_model.named_parameters())
    score_tensors: Dict[str, torch.Tensor] = {}

    with torch.no_grad():
        for param_name, base_param in base_named.items():
            # Only floating-point parameters participate in delta-space updates.
            if not torch.is_floating_point(base_param.data):
                continue

            base_fp32 = base_param.data.detach().to(torch.float32)
            if_fp32 = if_named[param_name].data.detach().to(torch.float32)
            delta_if = if_fp32 - base_fp32

            # Coordinate-wise grad-norm proxy for sparse masking.
            score_tensors[param_name] = delta_if.abs().cpu()

    return score_tensors


def count_total_elements(score_tensors: Mapping[str, torch.Tensor]) -> int:
    """Count total scalar coordinates across score tensors.

    Args:
        score_tensors: Parameter-score mapping.

    Returns:
        Total scalar coordinate count.
    """

    total_numel = 0
    for score_tensor in score_tensors.values():
        total_numel += int(score_tensor.numel())
    return int(total_numel)


def sample_global_score_values(
    score_tensors: Mapping[str, torch.Tensor],
    total_numel: int,
    sample_size: int,
    seed: int,
) -> np.ndarray:
    """Sample score values uniformly over global coordinate index space.

    Why this implementation exists:
        Fully concatenating all model coordinates into one giant vector is memory-heavy.
        This index-based streaming sample estimates global quantiles efficiently.

    Args:
        score_tensors: Parameter-score mapping.
        total_numel: Total number of scalar coordinates.
        sample_size: Number of sampled coordinates for threshold estimation.
        seed: RNG seed for reproducible sampling.

    Returns:
        1D NumPy array of sampled score values.
    """

    if total_numel <= 0 or sample_size <= 0:
        return np.zeros(0, dtype=np.float32)

    effective_sample_size = int(min(sample_size, total_numel))
    rng = np.random.default_rng(seed=seed)

    # Draw global coordinate indices with replacement, then stream through tensors once.
    sampled_global_indices = np.sort(
        rng.integers(low=0, high=total_numel, size=effective_sample_size, dtype=np.int64)
    )

    sampled_values = np.empty(effective_sample_size, dtype=np.float32)
    write_cursor = 0
    tensor_offset = 0

    for score_tensor in score_tensors.values():
        score_flat = score_tensor.detach().to(torch.float32).reshape(-1)
        tensor_numel = int(score_flat.numel())

        left = np.searchsorted(sampled_global_indices, tensor_offset, side='left')
        right = np.searchsorted(sampled_global_indices, tensor_offset + tensor_numel, side='left')

        if right > left:
            local_indices = sampled_global_indices[left:right] - tensor_offset
            local_index_tensor = torch.from_numpy(local_indices.astype(np.int64))
            local_values = score_flat.index_select(dim=0, index=local_index_tensor)

            next_cursor = write_cursor + (right - left)
            sampled_values[write_cursor:next_cursor] = local_values.cpu().numpy()
            write_cursor = next_cursor

        tensor_offset += tensor_numel

    if write_cursor != effective_sample_size:
        raise RuntimeError(
            f'Sampling bookkeeping mismatch: expected {effective_sample_size}, got {write_cursor}'
        )

    return sampled_values


def estimate_global_score_threshold(
    score_tensors: Mapping[str, torch.Tensor],
    keep_ratio: float,
    sample_size: int,
    seed: int,
) -> Tuple[float, int, int]:
    """Estimate a global score threshold that keeps the top ratio of coordinates.

    Args:
        score_tensors: Parameter-score mapping.
        keep_ratio: Target keep ratio in `(0, 1]`.
        sample_size: Number of sampled values used for quantile estimation.
        seed: RNG seed for sampling.

    Returns:
        Tuple `(threshold, total_numel, sampled_count)`.
    """

    if keep_ratio <= 0.0 or keep_ratio > 1.0:
        raise ValueError(f'keep_ratio must be in (0, 1], got {keep_ratio}')

    total_numel = count_total_elements(score_tensors=score_tensors)
    sampled_values = sample_global_score_values(
        score_tensors=score_tensors,
        total_numel=total_numel,
        sample_size=sample_size,
        seed=seed,
    )

    if sampled_values.size == 0:
        return 0.0, int(total_numel), int(sampled_values.size)

    # Top keep_ratio corresponds to quantile (1 - keep_ratio).
    threshold = float(np.quantile(sampled_values, q=(1.0 - float(keep_ratio))))
    return threshold, int(total_numel), int(sampled_values.size)


def apply_sparse_if_update_inplace(
    base_model: AutoModelForCausalLM,
    if_model: AutoModelForCausalLM,
    score_tensors: Mapping[str, torch.Tensor],
    score_threshold: float,
) -> Dict[str, Any]:
    """Apply sparse IF task-vector update to the base model in-place.

    Update rule:
        theta_sparse = theta_base + 1[score >= threshold] * (theta_if - theta_base)

    Args:
        base_model: Base model to overwrite with sparse-updated weights.
        if_model: IF model that provides dense IF delta.
        score_tensors: Parameter-score mapping used for coordinate masking.
        score_threshold: Global threshold for top-k coordinate retention.

    Returns:
        Summary dictionary with threshold and realized retention statistics.
    """

    validate_parameter_compatibility(base_model=base_model, if_model=if_model)

    base_named = dict(base_model.named_parameters())
    if_named = dict(if_model.named_parameters())

    kept_elements = 0
    total_elements = 0

    with torch.no_grad():
        for param_name, base_param in base_named.items():
            if not torch.is_floating_point(base_param.data):
                continue

            if param_name not in score_tensors:
                raise ValueError(f'Missing score tensor for parameter: {param_name}')

            score = score_tensors[param_name].detach().to(torch.float32)
            if tuple(score.shape) != tuple(base_param.shape):
                raise ValueError(
                    f"Score shape mismatch for '{param_name}': "
                    f"score={tuple(score.shape)} vs model={tuple(base_param.shape)}"
                )

            base_fp32 = base_param.data.detach().to(torch.float32)
            if_fp32 = if_named[param_name].data.detach().to(torch.float32)
            delta_if = if_fp32 - base_fp32

            # Keep only high-score coordinates and restore all others to base.
            sparse_mask = score >= float(score_threshold)
            sparse_update = delta_if * sparse_mask.to(torch.float32)
            merged_tensor = base_fp32 + sparse_update
            base_param.data.copy_(merged_tensor.to(base_param.dtype))

            kept_elements += int(sparse_mask.sum().item())
            total_elements += int(sparse_mask.numel())

    return {
        'score_threshold': float(score_threshold),
        'kept_elements': int(kept_elements),
        'total_elements': int(total_elements),
        'realized_keep_ratio': float(kept_elements / max(total_elements, 1)),
    }


def keep_ratio_to_tag(keep_ratio: float) -> str:
    """Convert keep ratio to a filesystem-safe tag string.

    Args:
        keep_ratio: Keep ratio in `(0, 1]`.

    Returns:
        Tag like `top_1pct` or `top_10pct`.
    """

    pct_value = keep_ratio * 100.0
    pct_str = f'{pct_value:.3f}'.rstrip('0').rstrip('.')
    return f"top_{pct_str.replace('.', 'p')}pct"


def save_sparse_if_checkpoint(
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    output_dir: Path,
    metadata: Mapping[str, Any],
) -> None:
    """Save sparse IF checkpoint and metadata.

    Args:
        model: Sparse-updated model.
        tokenizer: Tokenizer saved together with the model.
        output_dir: Output checkpoint directory.
        metadata: JSON-serializable metadata payload.

    Returns:
        None. Artifacts are written to `output_dir`.
    """

    output_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(output_dir, safe_serialization=True)
    tokenizer.save_pretrained(output_dir)
    save_json(metadata, output_dir / 'merge_metadata.json')

In [ ]:
merge_dtype = resolve_torch_dtype(RUNTIME.model_dtype_name)

# Load IF model once because it is reused for all sparse variants.
if_model_for_delta, _ = load_causal_lm(
    model_name_or_path=RUNTIME.if_model_path,
    torch_dtype=merge_dtype,
    device='cpu',
)

# Build score tensors from IF task-vector magnitudes against the base anchor.
base_anchor_for_scores, _ = load_causal_lm(
    model_name_or_path=RUNTIME.base_model_id,
    torch_dtype=merge_dtype,
    device='cpu',
)
score_tensors = build_if_task_vector_gradnorm_scores(
    base_model=base_anchor_for_scores,
    if_model=if_model_for_delta,
)
del base_anchor_for_scores
gc.collect()

sparse_run_rows = []

for keep_ratio in SPARSE_KEEP_RATIOS:
    threshold, total_numel, sampled_count = estimate_global_score_threshold(
        score_tensors=score_tensors,
        keep_ratio=keep_ratio,
        sample_size=THRESHOLD_SAMPLE_SIZE,
        seed=RUNTIME.seed,
    )

    # Reload a fresh base model so each sparse variant starts from the same anchor.
    sparse_model, sparse_tokenizer = load_causal_lm(
        model_name_or_path=RUNTIME.base_model_id,
        torch_dtype=merge_dtype,
        device='cpu',
    )

    sparse_summary = apply_sparse_if_update_inplace(
        base_model=sparse_model,
        if_model=if_model_for_delta,
        score_tensors=score_tensors,
        score_threshold=threshold,
    )

    output_tag = keep_ratio_to_tag(keep_ratio=keep_ratio)
    output_dir = SPARSE_OUTPUT_ROOT / f'if_delta_sparse_{output_tag}'

    metadata = {
        'created_at': now_iso(),
        'method': 'if_delta_sparse_by_task_vector_gradnorm_topk',
        'formula': 'theta_sparse = theta_base + 1[|Delta_if| >= threshold] * Delta_if',
        'score_definition': '|Delta_if| where Delta_if = theta_if - theta_base',
        'base_model_id': str(RUNTIME.base_model_id),
        'if_model_path': str(RUNTIME.if_model_path),
        'keep_ratio': float(keep_ratio),
        'sample_size_for_threshold': int(THRESHOLD_SAMPLE_SIZE),
        'threshold_estimation_total_numel': int(total_numel),
        'threshold_estimation_sampled_count': int(sampled_count),
        'sparse_summary': sparse_summary,
    }

    save_sparse_if_checkpoint(
        model=sparse_model,
        tokenizer=sparse_tokenizer,
        output_dir=output_dir,
        metadata=metadata,
    )

    sparse_run_rows.append(
        {
            'keep_ratio': float(keep_ratio),
            'score_threshold': float(sparse_summary['score_threshold']),
            'realized_keep_ratio': float(sparse_summary['realized_keep_ratio']),
            'kept_elements': int(sparse_summary['kept_elements']),
            'total_elements': int(sparse_summary['total_elements']),
            'output_dir': str(output_dir),
        }
    )

    print(
        f"Saved sparse IF checkpoint | keep_ratio={keep_ratio:.4f} "
        f"| realized={sparse_summary['realized_keep_ratio']:.6f} "
        f"| threshold={sparse_summary['score_threshold']:.6e} "
        f"| path={output_dir}"
    )

    del sparse_model
    del sparse_tokenizer
    gc.collect()

summary_payload = {
    'created_at': now_iso(),
    'runtime': asdict(RUNTIME),
    'method': 'if_delta_sparse_by_task_vector_gradnorm_topk',
    'keep_ratios': [float(ratio) for ratio in SPARSE_KEEP_RATIOS],
    'sample_size_for_threshold': int(THRESHOLD_SAMPLE_SIZE),
    'runs': sparse_run_rows,
}
save_json(summary_payload, SUMMARY_PATH)

display(pd.DataFrame(sparse_run_rows))
print(f'Saved sparse IF run summary: {SUMMARY_PATH}')

del if_model_for_delta
del score_tensors
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()